# LAB05 — Sequence Models (IMDb sentiment)
So sánh TF-IDF+LR vs Embedding+RNN/LSTM/GRU, thí nghiệm max_length, probe long-range dependency.
Chạy full ~1-2 phút trên GPU (CPU chậm hơn nhiều). Seed=42 mọi nơi.

In [ ]:
import re, time
import numpy as np, matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
SEED=42
def tok(s): return re.findall(r"[a-z']+", s.lower())
ds = load_dataset('stanfordnlp/imdb')
tr_texts, tr_labels = ds['train']['text'], np.array(ds['train']['label'])
te_texts, te_labels = ds['test']['text'], np.array(ds['test']['label'])
print(len(tr_texts), len(te_texts), tr_labels.mean(), te_labels.mean())
lens = np.array([len(tok(s)) for s in tr_texts])
print(f'min={lens.min()} med={np.median(lens)} mean={lens.mean():.1f} max={lens.max()}')
for ml in [30,50,100,200]:
    print(f'ml={ml}: tokens kept={np.minimum(lens,ml).sum()/lens.sum()*100:.1f}% docs full={(lens<=ml).mean()*100:.1f}%')
plt.hist(lens, bins=60); plt.axvline(200, c='r'); plt.title('Review length (words), red=max_len=200'); plt.show()


In [ ]:
# Core: RNN cell tu dau + verify (rnn_cell.py)
from rnn_cell import SimpleRNNCell
import torch
c = SimpleRNNCell(2,2,seed=0)
c.W_xh=np.array([[0.5,0.2],[0.1,0.4]]); c.W_hh=np.array([[0.3,0.1],[0.2,0.5]]); c.b_h=np.zeros(2)
print('h_t =', c.forward(np.array([1.,0.]), np.array([0.5,0.2])), '(expect ~[0.585, 0.291])')


In [ ]:
# Vocab (train-only) + loaders + model + train loop chung
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
torch.manual_seed(SEED); DEV='cuda' if torch.cuda.is_available() else 'cpu'
X_tr,X_va,y_tr,y_va = train_test_split(list(tr_texts), tr_labels, test_size=0.2, random_state=SEED, stratify=tr_labels)
cnt=Counter()
for s in X_tr: cnt.update(tok(s))
words=[w for w,c in cnt.most_common() if c>=5][:10000-2]
w2i={w:i+2 for i,w in enumerate(words)}; V=len(w2i)+2
print('vocab:',V)
def encode(s,ml):
    ids=[w2i.get(w,1) for w in tok(s)][:ml]; return ids+[0]*(ml-len(ids))
def loader(texts,labels,ml,bs=64,sh=True):
    return DataLoader(TensorDataset(torch.tensor([encode(s,ml) for s in texts]),torch.tensor(np.array(labels),dtype=torch.float32)),batch_size=bs,shuffle=sh)
class SeqClf(nn.Module):
    def __init__(s,arch):
        super().__init__(); s.emb=nn.Embedding(V,100,padding_idx=0)
        s.rnn={'rnn':nn.RNN,'lstm':nn.LSTM,'gru':nn.GRU}[arch](100,128,batch_first=True); s.fc=nn.Linear(128,1)
    def forward(s,x):
        _,h=s.rnn(s.emb(x)); h=h[0] if isinstance(h,tuple) else h; return s.fc(h[-1]).squeeze(1)
def train(arch,ml,epochs=5):
    m=SeqClf(arch).to(DEV); opt=torch.optim.Adam(m.parameters(),lr=1e-3); crit=nn.BCEWithLogitsLoss()
    H={'tr':[],'va':[]}; t0=time.time()
    for ep in range(epochs):
        m.train(); tl=n=0
        for xb,yb in loader(X_tr,y_tr,ml):
            xb,yb=xb.to(DEV),yb.to(DEV); opt.zero_grad(); l=crit(m(xb),yb); l.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); tl+=l.item()*len(xb); n+=len(xb)
        m.eval(); vl=nv=0
        with torch.no_grad():
            for xb,yb in loader(X_va,y_va,ml,sh=False):
                xb,yb=xb.to(DEV),yb.to(DEV); vl+=crit(m(xb),yb).item()*len(xb); nv+=len(xb)
        H['tr'].append(tl/n); H['va'].append(vl/nv); print(f'{arch}/{ml} ep{ep+1}: tr={tl/n:.4f} va={vl/nv:.4f}')
    dt=time.time()-t0; m.eval(); ps=[]
    with torch.no_grad():
        for xb,_ in loader(te_texts,te_labels,ml,sh=False): ps.append(torch.sigmoid(m(xb.to(DEV))).cpu().numpy())
    prob=np.concatenate(ps); pred=(prob>=0.5).astype(int)
    print(f"== {arch} ml={ml} {dt:.1f}s acc={accuracy_score(te_labels,pred):.4f} F1={f1_score(te_labels,pred):.4f}")
    return m,prob,pred,H,dt


In [ ]:
# Baseline LAB04 + 3 recurrent models (ml=200)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
vec=TfidfVectorizer(max_features=10000, token_pattern=r'(?u)\b\w+\b')
bl=LogisticRegression(max_iter=1000).fit(vec.fit_transform(X_tr),y_tr)
pb=bl.predict(vec.transform(te_texts))
print(f"BASELINE acc={accuracy_score(te_labels,pb):.4f} F1={f1_score(te_labels,pb):.4f}")
m_rnn,_,pr_rnn,H_rnn,_=train('rnn',200)
m_lstm,_,_,H_lstm,_=train('lstm',200)
m_gru,prob_gru,pred_gru,H_gru,_=train('gru',200)


RNN loss kẹt ~0.69 (vanishing), LSTM/GRU học được, nhưng baseline TF-IDF (F1 0.883) vẫn cao nhất — đa số review tách được bằng từ vựng.

In [ ]:
# Exp max_length (GRU) + curves + confusion matrix
_,_,_,H_g50,_=train('gru',50); _,_,_,H_g100,_=train('gru',100)
plt.plot(H_rnn['va'],label='rnn'); plt.plot(H_lstm['va'],label='lstm'); plt.plot(H_gru['va'],label='gru')
plt.title('Val loss (ml=200)'); plt.xlabel('epoch'); plt.legend(); plt.show()
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(te_labels,pred_gru); print(cm)
plt.imshow(cm,cmap='Blues'); plt.xticks([0,1],['pred neg','pred pos']); plt.yticks([0,1],['true neg','true pos'])
plt.title('GRU-200 test CM'); plt.savefig('confusion_matrix.png',dpi=150); plt.show()


In [ ]:
# Probe vi tri keyword + application 4 loai review (GRU-200)
def pg(s,ml=200):
    m_gru.eval()
    with torch.no_grad(): return float(torch.sigmoid(m_gru(torch.tensor([encode(s,ml)]).to(DEV))).cpu().numpy()[0])
def pb2(s): return float(bl.predict_proba(vec.transform([s]))[0][1])
F='with many scenes about daily life and ordinary events'
for n,s,y in [('END-pos',f'The film was long {F} {F}, but the ending was excellent .',1),('END-neg',f'The film was long {F} {F}, but the ending was terrible .',0),('MID-pos',f'The film had {F}, an excellent middle act, then {F} again .',1),('MID-neg',f'The film had {F}, a terrible middle act, then {F} again .',0),('BEGIN-pos',f'Excellent from the very start, yet the film then had {F} {F} .',1),('BEGIN-neg',f'Terrible from the very start, yet the film then had {F} {F} .',0)]:
    print(n,'true=',y,'Ppos=',round(pg(s),4))
for n,s in [('short','I really enjoyed this movie.'),('negation','This movie is not good. I did not like it at all.'),('contrast','I thought the movie would be terrible, but it was surprisingly good.')]:
    print(n,f'BL={pb2(s):.4f} GRU={pg(s):.4f}')


Kết luận: negation cục bộ GRU làm tốt (0.104), contrast gãy cả 2 model (0.289 vs 0.329). Probe MID khó nhất (margin 0.10) — xem error_analysis.md và reflection.md (RNN → Attention).